## 1. 파일 로드

In [ ]:
import pandas as pd
import numpy as np

# 리뷰 DF (챗봇 API 분석 결과)
컬리챗봇 = pd.read_csv('[최종]컬리_챗봇api분석.csv')
곰곰챗봇 = pd.read_csv('[최종]곰곰_챗봇api분석.csv', encoding='utf-8-sig')

# 제품 DF
컬리제품 = pd.read_csv('제품DF_컬리.csv')
곰곰제품 = pd.read_csv('[최종]곰곰_제품DF.csv')

## 2. 리뷰 DF 정제

### 2-1. 컬리 리뷰 DF

In [ ]:
# ── 결측 행 삭제 ──────────────────────────────────────────────────
# 리뷰텍스트 결측 행 삭제 후 인덱스 재정렬
컬리챗봇 = 컬리챗봇.dropna(subset=['리뷰텍스트']).reset_index(drop=True)

In [ ]:
# ── 키워드 열 오류 수정 (1) : 품질 → 배송 이동 ───────────────────
# 챗봇 API가 배송 키워드를 품질 열에 잘못 입력한 경우 수동 수정
target_indices = [
    971, 1423, 1811, 2915, 3904, 6268, 9113, 9779, 18009, 21853,
    27150, 29430, 29908, 36271, 36896, 39159, 39745, 40270, 40559,
    42825, 42999, 43853, 46556
]
컬리챗봇.loc[target_indices, '배송'] = 컬리챗봇.loc[target_indices, '품질']
컬리챗봇.loc[target_indices, '품질'] = '키워드 없음'

In [ ]:
# ── 키워드 열 오류 수정 (2) : 품질 → 배송 이동 (조건 기반) ──────
# 배송 열이 결측이면서 품질 열에 '배송' 단어가 포함된 경우
mask = 컬리챗봇['배송'].isnull() & 컬리챗봇['품질'].str.contains('배송', na=False)
컬리챗봇.loc[mask, '배송'] = 컬리챗봇.loc[mask, '품질']
컬리챗봇.loc[mask, '품질'] = '키워드 없음'
print(f'처리된 행 개수: {mask.sum()}개')

In [ ]:
# ── 키워드 열 오류 수정 (3) : 품질 → 구성/가성비 이동 ───────────
target_indices = [3166, 43389]
컬리챗봇.loc[target_indices, '구성/가성비'] = 컬리챗봇.loc[target_indices, '품질']
컬리챗봇.loc[target_indices, '품질'] = '키워드 없음'

In [ ]:
# ── 잔여 결측치 → '키워드 없음' 일괄 처리 ───────────────────────
컬리챗봇['품질'] = 컬리챗봇['품질'].fillna('키워드 없음')
컬리챗봇['배송'] = 컬리챗봇['배송'].fillna('키워드 없음')

# 리뷰텍스트 결측 재확인 후 인덱스 재정렬
컬리챗봇 = 컬리챗봇.dropna(subset=['리뷰텍스트']).reset_index(drop=True)

컬리챗봇.to_csv('[0325]컬리_챗봇api분석_정제.csv', index=False)

### 2-2. 곰곰 리뷰 DF

In [ ]:
# ── 결측 행 삭제 ──────────────────────────────────────────────────
# 리뷰텍스트 결측 행 삭제
곰곰챗봇 = 곰곰챗봇.dropna(subset=['리뷰텍스트']).reset_index(drop=True)

# 감성점수 결측 행 삭제
곰곰챗봇 = 곰곰챗봇.dropna(subset=['감성점수']).reset_index(drop=True)

# 감성점수 열 '키워드 없음' 값 제거 및 데이터 타입 변환
곰곰챗봇 = 곰곰챗봇[곰곰챗봇['감성점수'] != '키워드 없음']
곰곰챗봇['감성점수'] = pd.to_numeric(곰곰챗봇['감성점수'], errors='coerce').astype(int)

In [ ]:
# ── 키워드 열 오류 수정 (1) : 품질 → 배송 이동 (조건 기반) ──────
mask = 곰곰챗봇['배송'].isnull() & 곰곰챗봇['품질'].str.contains('배송', na=False)
곰곰챗봇.loc[mask, '배송'] = 곰곰챗봇.loc[mask, '품질']
곰곰챗봇.loc[mask, '품질'] = '키워드 없음'
print(f'처리된 행 개수: {mask.sum()}개')

In [ ]:
# ── 키워드 열 오류 수정 (2) : 품질 → 구성/가성비 이동 (조건 기반)
mask2 = 곰곰챗봇['구성/가성비'].isnull() & 곰곰챗봇['품질'].str.contains('가성비', na=False)
곰곰챗봇.loc[mask2, '구성/가성비'] = 곰곰챗봇.loc[mask2, '품질']
곰곰챗봇.loc[mask2, '품질'] = '키워드 없음'
print(f'처리된 행 개수: {mask2.sum()}개')

In [ ]:
# ── 잔여 결측치 → '키워드 없음' 일괄 처리 ───────────────────────
곰곰챗봇['구성/가성비'] = 곰곰챗봇['구성/가성비'].fillna('키워드 없음')

곰곰챗봇.to_csv('[0325]곰곰_챗봇api분석_정제.csv', index=False)
print('곰곰 리뷰 DF 정제 완료:', 곰곰챗봇.shape)

### 2-3. 통합 리뷰 DF 생성

In [ ]:
통합챗봇DF = pd.concat([컬리챗봇, 곰곰챗봇], ignore_index=True)

print(f'컬리챗봇 행 개수: {len(컬리챗봇)}')
print(f'곰곰챗봇 행 개수: {len(곰곰챗봇)}')
print(f'최종 통합 행 개수: {len(통합챗봇DF)}')
print(통합챗봇DF['브랜드'].value_counts())

통합챗봇DF.to_csv('[0325]통합_챗봇api분석_정제.csv', index=False)

## 3. 제품 DF 정제 — 곰곰

In [ ]:
# ── 불필요 열 삭제 및 열 순서 정렬 ──────────────────────────────
곰곰제품 = 곰곰제품.drop(columns=['원본URL'])

new_column_order = [
    '상품ID', '브랜드', '순위', '분류', '상품명',
    '할인율', '할인가', '총용량(g)', '단위가격(100g)', '리뷰수', '평균별점'
]
곰곰제품 = 곰곰제품[new_column_order]

In [ ]:
# ── 분류명 통일 ───────────────────────────────────────────────────
곰곰제품['분류'] = 곰곰제품['분류'].replace({'과일': '과일·채소', '햄': '가공육'})
print(곰곰제품['분류'].value_counts())

In [ ]:
# ── 리뷰수 데이터 타입 변환 (콤마 제거 후 float 변환) ─────────────
곰곰제품['리뷰수'] = 곰곰제품['리뷰수'].astype(str).str.replace(',', '').astype(float)

곰곰제품.to_csv('[찐]곰곰_제품DF.csv', index=False)
print('곰곰 제품 DF 정제 완료:', 곰곰제품.shape)

## 4. 제품 DF 정제 — 컬리

### 4-1. 기본 정제

In [ ]:
# ── 불필요 열 삭제 및 데이터 타입 변환 ───────────────────────────
컬리제품 = 컬리제품.drop(columns=['컬리상품ID'])
컬리제품['할인율'] = 컬리제품['할인율'].astype(float)

print(컬리제품.info())

### 4-2. 소분류 추출 (Vertex AI Gemini API 활용)

- 컬리는 대분류 카테고리로 수집되었으므로, 상품명과 대분류를 입력으로 소분류를 자동 추출
- 사용 모델: gemini-2.5-flash-lite

In [ ]:
import os
import json
import time
import vertexai
from google.oauth2 import service_account
from vertexai.generative_models import GenerativeModel, GenerationConfig
from typing import Any

KEY_PATH = ' ' # 채워넣기
project_id = ' ' # 채워넣기
location = 'us-central1'
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

vertexai.init(project=project_id, location=location, credentials=credentials)
model = GenerativeModel('gemini-2.5-flash-lite')

In [ ]:
# 지침 전달
guide = '''
# 상품명을 보고 대분류에서 소분류 추출하기

## 상품명
- 상품명이 주어집니다. 마켓컬리의 pb브랜드 상품 중 식품군에 해당되는 상품입니다.

## 대분류
- 분류가 주어집니다. 이는 '대분류'라고 칭합니다.

## 소분류 추출
- 상품명을 읽고 상품의 종류를 파악합니다.
- 해당 상품이 속하는 '대분류'에서 '소분류'를 추출합니다. (예: 대분류 - 정육·가공육·달걀 -> 소분류 - 달걀 )
- 대분류(예: 정육·가공육·달걀)에 속하는 단어(예: 정육, 가공육, 달걀) 중에서만 추출할 것. 그 외의 단어는 추출하지 않습니다.
- 대분류명이 하나의 단어로 이루어진 경우 별도로 소분류를 추출하지 않습니다. (예: 대분류 - 베이커리 -> 소분류 - 베이커리)

## 답변형식
- JSON 형식으로 답변.
- 소분류명만 반환
- 추가정보 언급 금지.
'''

In [ ]:
# 함수 정의
def apply_gemini_in_batches(
    df,
    model_obj, # Vertex AI 모델 객체 추가
    guide,
    name_col = '상품명',
    cat_col='분류',
    save_every=50,
    backup_path='kurly_cat_backup.csv',
    sleep_sec=0.3
):
    df = df.copy()
    target_col = '소분류'
    status_col = '처리완료'

    # 초기 컬럼 설정
    if target_col not in df.columns:
        df[target_col] = pd.NA
    if status_col not in df.columns:
        df[status_col] = False

    # 1. 백업 파일 로드 및 기존 결과 반영
    if os.path.exists(backup_path):
        backup_df = pd.read_csv(backup_path)
        if len(backup_df) == len(df):
            # 처리 완료된 행만 기존 df에 업데이트
            done_mask = backup_df[status_col] == True
            df.loc[done_mask, [target_col, status_col]] = backup_df.loc[done_mask, [target_col, status_col]].values
            print(f"백업 로드 완료: {done_mask.sum()}건이 이미 처리되었습니다.")

    pending_mask = df[status_col] != True
    pending_idx = df.index[pending_mask].tolist()
    total = len(pending_idx)

    if total == 0:
        print("모든 데이터가 이미 처리되었습니다.")
        return df

    print(f'처리 시작 | 남은 데이터: {total}')
    batch_updates = []

    for i, idx in enumerate(pending_idx, start=1):
        product_name = df.at[idx, name_col]
        main_cat = df.at[idx, cat_col]

        # 프롬프트 구성
        prompt = f"[Instruction]\n{guide}\n\n[Input]\n상품명: {product_name}\n대분류: {main_cat}\n\n[Response]"

        try:
            # 2. Vertex AI 모델 호출
            response = model_obj.generate_content(
                prompt,
                generation_config={
                    "max_output_tokens": 512,
                    "temperature": 0.0, # 분류 정확도를 위해 0으로 설정
                    "response_mime_type": "application/json" # JSON 출력 강제 (모델 지원 시)
                }
            )

            result_text = response.text.replace("```json", "").replace("```", "").strip()
            result_data = json.loads(result_text)

            # 3. 데이터 추출 방어 로직 (딕셔너리 또는 단일 값 처리)
            if isinstance(result_data, dict):
                sub_cat = list(result_data.values())[0]
            elif isinstance(result_data, list):
                sub_cat = result_data[0]
            else:
                sub_cat = result_data

            batch_updates.append({
                'idx': idx,
                '소분류': sub_cat,
                '처리완료': True
            })

        except Exception as e:
            print(f"Error at index {idx}: {e}")
            batch_updates.append({
                'idx': idx,
                '소분류': pd.NA,
                '처리완료': False
            })

        # API 할당량 준수를 위한 대기
        time.sleep(sleep_sec)

        # 4. 중간 저장 (배치 단위)
        if (i % save_every == 0) or (i == total):
            update_df = pd.DataFrame(batch_updates).set_index('idx')
            df.loc[update_df.index, [target_col, status_col]] = update_df[[target_col, status_col]]
            df.to_csv(backup_path, index=False, encoding='utf-8-sig')
            print(f'진행률 {i}/{total} | 백업 완료')
            batch_updates = []

    print('전체 작업 완료')
    return df

In [ ]:
# 작업 실행
result_df = apply_gemini_in_batches(
    df=df,
    model_obj=model,
    guide=guide,
    name_col='상품명',
    cat_col='분류',
    save_every=30,
    backup_path='kurly_cat_backup.csv',
    sleep_sec=0.5
)

In [ ]:
# ── 소분류 오류 수동 수정 ─────────────────────────────────────────
result_df.loc[[157], '소분류'] = '수산'   # 해산 → 수산 수정
result_df.loc[[149], '소분류'] = '수산'

In [ ]:
# ── 상품ID 기준 중복 제거 ─────────────────────────────────────────
result_df = result_df.drop_duplicates(subset='상품ID', keep='first').reset_index(drop=True)
print(f'중복 제거 후 행 개수: {len(result_df)}')

In [ ]:
# ── 비식품군 행 수동 삭제 ─────────────────────────────────────────
result_df = result_df.drop(index=[212, 213]).reset_index(drop=True)

result_df.to_csv('[최종]제품DF_컬리_1차정제(분류명,중복).csv', index=False)

## 5. 파생변수 생성

### 5-1. 공통 함수 정의

In [ ]:
# ── 감성 분류 함수 (4~5점: 긍정 / 1~3점: 부정) ───────────────────
def classify_sentiment(score):
    return '긍정' if score >= 4 else '부정'

# ── 최빈 키워드 추출 함수 ('키워드 없음' 제외) ────────────────────
def get_top_keyword(series):
    valid = series[series != '키워드 없음'].dropna()
    return valid.mode()[0] if not valid.empty else '키워드 없음'

# ── 상품별 파생변수 생성 함수 ─────────────────────────────────────
def get_refined_review_stats(group):
    total_count = len(group)
    if total_count == 0:
        return pd.Series()

    pos_ratio = (group['감성점수'] >= 4).sum() / total_count
    neg_ratio = (group['감성점수'] <= 3).sum() / total_count
    low_star_ratio = (group['별점'] <= 2).sum() / total_count

    def count_valid_mentions(series):
        valid_condition = series.notna() & (series.astype(str).str.strip() != '키워드 없음')
        return valid_condition.sum()

    quality_mentions = count_valid_mentions(group['품질'])
    value_mentions = count_valid_mentions(group['구성/가성비'])
    delivery_mentions = count_valid_mentions(group['배송'])

    # 최근 30일 리뷰 비율 (기준일: 2026-03-24)
    target_date = pd.to_datetime('2026-03-24') - pd.Timedelta(days=30)
    recent_mentions = (pd.to_datetime(group['날짜']) >= target_date).sum()

    return pd.Series({
        '긍정리뷰비율': round(pos_ratio, 2),
        '부정리뷰비율': round(neg_ratio, 2),
        '품질언급비율': round(quality_mentions / total_count, 2),
        '가성비언급비율': round(value_mentions / total_count, 2),
        '배송언급비율': round(delivery_mentions / total_count, 2),
        '저평점비율': round(low_star_ratio, 2),
        '최근 30일 리뷰비율': round(recent_mentions / total_count, 2)
    })

Index(['상품ID', '컬리상품ID', '브랜드', '순위', '분류', '상품명', '할인율', '할인가', '총용량(g)',
       '단위가격(100g)', '리뷰수', '평균별점', '소분류', '처리완료'],
      dtype='object')

### 5-2. 컬리 파생변수 생성

In [ ]:
# ── 소분류 병합 ───────────────────────────────────────────────────
소분류_정제 = result_df.drop_duplicates(subset='상품ID', keep='first')
컬리제품1 = pd.merge(컬리제품, 소분류_정제[['상품ID', '소분류']], on='상품ID', how='left').reset_index(drop=True)
print(f'소분류 병합 후 행 개수: {len(컬리제품1)}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 313 entries, 0 to 312
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   상품ID        313 non-null    int64  
 1   컬리상품ID      313 non-null    int64  
 2   브랜드         313 non-null    object 
 3   순위          313 non-null    int64  
 4   분류          313 non-null    object 
 5   상품명         313 non-null    object 
 6   할인율         313 non-null    int64  
 7   할인가         313 non-null    int64  
 8   총용량(g)      239 non-null    float64
 9   단위가격(100g)  239 non-null    float64
 10  리뷰수         313 non-null    float64
 11  평균별점        313 non-null    float64
 12  소분류         313 non-null    object 
 13  처리완료        313 non-null    bool   
dtypes: bool(1), float64(4), int64(5), object(4)
memory usage: 32.2+ KB


In [ ]:
# ── 감성분류 열 생성 ──────────────────────────────────────────────
컬리챗봇['감성분류'] = 컬리챗봇['감성점수'].apply(classify_sentiment)

# ── 상품별 최빈 키워드 추출 및 병합 ──────────────────────────────
results = []
for pid, group in 컬리챗봇.groupby('상품ID'):
    pos_group = group[group['감성분류'] == '긍정']
    neg_group = group[group['감성분류'] == '부정']
    results.append({
        '상품ID': pid,
        '품질(긍정)': get_top_keyword(pos_group['품질']),
        '품질(부정)': get_top_keyword(neg_group['품질']),
        '구성/가성비(긍정)': get_top_keyword(pos_group['구성/가성비']),
        '구성/가성비(부정)': get_top_keyword(neg_group['구성/가성비']),
        '배송(긍정)': get_top_keyword(pos_group['배송']),
        '배송(부정)': get_top_keyword(neg_group['배송'])
    })

df_keywords = pd.DataFrame(results)
컬리제품1 = pd.merge(컬리제품1, df_keywords, on='상품ID', how='left')
print(f'최빈 키워드 병합 후 행 개수: {len(컬리제품1)}')

In [ ]:
# ── 기타 파생변수 생성 ────────────────────────────────────────────
stats_df = 컬리챗봇.groupby('상품ID').apply(get_refined_review_stats).reset_index()
컬리제품1 = pd.merge(컬리제품1, stats_df, on='상품ID', how='left')

In [ ]:
# ── 소분류 결측 행 삭제 (생수·음료 분류) ─────────────────────────
컬리제품1 = 컬리제품1.dropna(subset=['소분류']).reset_index(drop=True)
print(f'소분류 결측 삭제 후 행 개수: {len(컬리제품1)}')

In [ ]:
# ── 파생변수 결측치 처리 ──────────────────────────────────────────
컬리제품DF_최종 = 컬리제품1.copy()
target_cols = ['긍정리뷰비율', '부정리뷰비율', '품질언급비율',
               '가성비언급비율', '배송언급비율', '저평점비율', '최근 30일 리뷰비율']

# 리뷰수 0인 경우 → 0으로 채우기
mask_zero = (컬리제품DF_최종['리뷰수'] == 0)
for col in target_cols:
    컬리제품DF_최종.loc[mask_zero, col] = 컬리제품DF_최종.loc[mask_zero, col].fillna(0)

# 리뷰수 0이 아닌 경우 → 해당 열 평균값으로 채우기
mask_exist = (컬리제품DF_최종['리뷰수'] > 0)
for col in target_cols:
    col_mean = 컬리제품DF_최종[col].mean()
    컬리제품DF_최종.loc[mask_exist, col] = 컬리제품DF_최종.loc[mask_exist, col].fillna(col_mean)

In [ ]:
# ── 분류명 통일 ───────────────────────────────────────────────────
컬리제품DF_최종['소분류'] = 컬리제품DF_최종['소분류'].replace({'과일': '과일·채소', '채소': '과일·채소'})

# ── 분류 열 정리 (기존 대분류 삭제 → 소분류를 '분류'로 rename) ───
컬리제품DF_최종 = 컬리제품DF_최종.drop(columns=['분류'])
컬리제품DF_최종 = 컬리제품DF_최종.rename(columns={'소분류': '분류'})

# ── 열 순서 재배치 ────────────────────────────────────────────────
new_column_order = [
    '상품ID', '브랜드', '순위', '분류', '상품명', '할인율', '할인가',
    '총용량(g)', '단위가격(100g)', '리뷰수', '평균별점', '긍정리뷰비율',
    '부정리뷰비율', '품질언급비율', '가성비언급비율', '배송언급비율',
    '저평점비율', '최근 30일 리뷰비율'
]
컬리제품DF_최종 = 컬리제품DF_최종[new_column_order]

In [ ]:
# ── 데이터 타입 정제 ──────────────────────────────────────────────
컬리제품DF_최종['총용량(g)'] = 컬리제품DF_최종['총용량(g)'].fillna(0).astype('int64')
컬리제품DF_최종['단위가격(100g)'] = 컬리제품DF_최종['단위가격(100g)'].fillna(0).astype('int64')
컬리제품DF_최종['상품ID'] = 컬리제품DF_최종['상품ID'].astype('int64')

In [ ]:
# ── 상품ID 기준 중복 제거 ─────────────────────────────────────────
# 삭제 전: 387행 → 삭제 후: 313행
컬리제품DF_중복제거 = 컬리제품DF_최종.drop_duplicates(subset=['상품ID'], keep='first')
print(f'중복 제거 후 행 개수: {len(컬리제품DF_중복제거)}')

In [ ]:
# ── 총용량·단위가격 결측 보완 (수기 수집 데이터 활용) ─────────────
new = pd.read_excel('컬리_총용량결측_수기정보수집_완료.xlsx')

# 단위가격 재계산
new['단위가격(100g)'] = ((new['할인가'] / new['총용량(g)']) * 100).round(0)

# 총용량 보완
capacity_map_vol = new.set_index('상품ID')['총용량(g)']
컬리제품DF_중복제거['총용량(g)'] = 컬리제품DF_중복제거['상품ID'].map(capacity_map_vol).fillna(컬리제품DF_중복제거['총용량(g)'])

# 단위가격 보완
capacity_map_price = new.set_index('상품ID')['단위가격(100g)']
컬리제품DF_중복제거['단위가격(100g)'] = 컬리제품DF_중복제거['상품ID'].map(capacity_map_price).fillna(컬리제품DF_중복제거['단위가격(100g)'])

# 데이터 타입 재정비
컬리제품DF_중복제거['총용량(g)'] = 컬리제품DF_중복제거['총용량(g)'].astype('int64')
컬리제품DF_중복제거['단위가격(100g)'] = 컬리제품DF_중복제거['단위가격(100g)'].astype('int64')

컬리제품DF_중복제거.to_csv('[찐]컬리_제품DF.csv', index=False)
print('컬리 제품 DF 파생변수 생성 완료:', 컬리제품DF_중복제거.shape)

### 5-3. 곰곰 파생변수 생성

In [ ]:
# ── 감성분류 열 생성 ──────────────────────────────────────────────
곰곰챗봇['감성분류'] = 곰곰챗봇['감성점수'].apply(classify_sentiment)

# ── 상품별 최빈 키워드 추출 및 병합 ──────────────────────────────
results = []
for pid, group in 곰곰챗봇.groupby('상품ID'):
    pos_group = group[group['감성분류'] == '긍정']
    neg_group = group[group['감성분류'] == '부정']
    results.append({
        '상품ID': pid,
        '품질(긍정)': get_top_keyword(pos_group['품질']),
        '품질(부정)': get_top_keyword(neg_group['품질']),
        '구성/가성비(긍정)': get_top_keyword(pos_group['구성/가성비']),
        '구성/가성비(부정)': get_top_keyword(neg_group['구성/가성비']),
        '배송(긍정)': get_top_keyword(pos_group['배송']),
        '배송(부정)': get_top_keyword(neg_group['배송'])
    })

df_keywords_곰곰 = pd.DataFrame(results)
곰곰제품1 = pd.merge(곰곰제품, df_keywords_곰곰, on='상품ID', how='left')
print(f'최빈 키워드 병합 후 행 개수: {len(곰곰제품1)}')

In [ ]:
# ── 기타 파생변수 생성 ────────────────────────────────────────────
stats_df_곰곰 = 곰곰챗봇.groupby('상품ID').apply(get_refined_review_stats).reset_index()
곰곰제품1 = pd.merge(곰곰제품1, stats_df_곰곰, on='상품ID', how='left')

곰곰제품1.to_csv('[0325]곰곰_제품DF.csv', index=False)
print('곰곰 제품 DF 파생변수 생성 완료:', 곰곰제품1.shape)

## 6. 최종 통합 DF 생성 및 저장

In [ ]:
# ── 통합 제품 DF ──────────────────────────────────────────────────
통합제품DF = pd.concat([컬리제품DF_중복제거, 곰곰제품1], ignore_index=True)

print(f'컬리 제품 행 개수: {len(컬리제품DF_중복제거)}')
print(f'곰곰 제품 행 개수: {len(곰곰제품1)}')
print(f'최종 통합 행 개수: {len(통합제품DF)}')
print(통합제품DF['브랜드'].value_counts())

통합제품DF.to_csv('[0325]통합_제품DF.csv', index=False)

print('[0325]통합_챗봇api분석_정제.csv')
print('  - [0325]통합_제품DF.csv')